# Platform Monitoring Queries

This notebook answers questions by analyzing pipeline execution metadata stored in the `pipeline_executions` Delta table. 

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("AnalyzePipelineMonitoring")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.session.timeZone", "UTC")

PIPELINE_EXECUTIONS_PATH = (
    "s3a://delta-data/metadata/pipeline_executions"
)

executions = (
    spark.read
    .format("delta")
    .load(PIPELINE_EXECUTIONS_PATH)
)

executions.createOrReplaceTempView("pipeline_executions")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/27 16:55:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/27 16:55:54 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Questions

### Which dataset fails validation most frequently?

In [2]:
spark.sql("""
    SELECT
        dataset,
        SUM(CASE WHEN validation_failure_count > 0 THEN 1 ELSE 0 END)
            AS executions_with_failures,
        SUM(COALESCE(rejected_records, 0))
            AS total_rejected_records,
        SUM(COALESCE(validation_failure_count, 0))
            AS total_rule_violations
    FROM pipeline_executions
    WHERE stage = 'silver'
    GROUP BY dataset
    ORDER BY executions_with_failures DESC, total_rejected_records DESC;
""").show(truncate=False)

26/09/27 16:55:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------------------------+----------------------+---------------------+
|dataset    |executions_with_failures|total_rejected_records|total_rule_violations|
+-----------+------------------------+----------------------+---------------------+
|taxi_trips |2                       |306939                |919779               |
|taxi_zones |1                       |1                     |1                    |
|air_quality|0                       |0                     |0                    |
|weather    |0                       |0                     |0                    |
+-----------+------------------------+----------------------+---------------------+



### Which dataset requires the longest processing time?

In [3]:
spark.sql("""
    SELECT
        stage,
        dataset,
        COUNT(*) AS execution_count,
        ROUND(AVG(duration_seconds), 3) AS average_duration_seconds,
        ROUND(MAX(duration_seconds), 3) AS longest_duration_seconds
    FROM pipeline_executions
    GROUP BY stage, dataset
    ORDER BY average_duration_seconds DESC
""").show(truncate=False)

+-----------------+--------------------------------+---------------+------------------------+------------------------+
|stage            |dataset                         |execution_count|average_duration_seconds|longest_duration_seconds|
+-----------------+--------------------------------+---------------+------------------------+------------------------+
|silver           |air_quality                     |2              |45.523                  |61.423                  |
|silver           |taxi_trips                      |2              |39.375                  |47.405                  |
|gold             |integrated_taxi_trips           |2              |25.573                  |35.411                  |
|bronze           |air_quality                     |2              |9.199                   |15.102                  |
|bronze           |taxi_trips                      |2              |6.212                   |6.735                   |
|analytics_refresh|average_trip_distance_by_weat

### How many records were rejected during each execution?

In [4]:
spark.sql("""
    SELECT
        pipeline_run_id,
        execution_id,
        stage,
        dataset,
        rejected_records,
        started_at
    FROM pipeline_executions
    ORDER BY rejected_records DESC, started_at DESC
""").show(100, truncate=False)

+------------------------------------+------------------------------------+-----------------+--------------------------------+----------------+--------------------------+
|pipeline_run_id                     |execution_id                        |stage            |dataset                         |rejected_records|started_at                |
+------------------------------------+------------------------------------+-----------------+--------------------------------+----------------+--------------------------+
|f33a0766-bd2d-4d85-a1df-c5436075ea5f|987a7c1e-532e-49b5-8603-e3cf4574d303|silver           |taxi_trips                      |194717          |2026-09-27 16:46:48.04597 |
|b2d2466b-ae95-4bfe-ade1-95d96c1b00c0|d7345fc0-6048-478e-af09-6fd769175a23|silver           |taxi_trips                      |112222          |2026-09-27 16:49:37.896103|
|f33a0766-bd2d-4d85-a1df-c5436075ea5f|e89d6782-ba26-435d-8d75-3c8ac11a899d|silver           |taxi_zones                      |1               |20

### How has processing time changed over multiple executions?

In [5]:
spark.sql("""
    SELECT
        stage,
        dataset,
        pipeline_run_id,
        started_at,
        duration_seconds
    FROM pipeline_executions
    ORDER BY stage, dataset, started_at
""").show(100, truncate=False)

+-----------------+--------------------------------+------------------------------------+--------------------------+----------------+
|stage            |dataset                         |pipeline_run_id                     |started_at                |duration_seconds|
+-----------------+--------------------------------+------------------------------------+--------------------------+----------------+
|analytics_refresh|air_quality_vs_demand           |f33a0766-bd2d-4d85-a1df-c5436075ea5f|2026-09-27 16:48:39.40459 |3.433           |
|analytics_refresh|air_quality_vs_demand           |b2d2466b-ae95-4bfe-ade1-95d96c1b00c0|2026-09-27 16:52:01.880019|5.577           |
|analytics_refresh|average_trip_distance_by_weather|f33a0766-bd2d-4d85-a1df-c5436075ea5f|2026-09-27 16:48:33.912058|3.895           |
|analytics_refresh|average_trip_distance_by_weather|b2d2466b-ae95-4bfe-ade1-95d96c1b00c0|2026-09-27 16:51:55.565161|6.098           |
|analytics_refresh|monthly_zone_demand             |f33a0766-b

In [6]:
spark.stop()